# ครั้งที่ 3 — การแทนข้อมูลตัวเลข (Python Notebook)
**51423364 Embedded System Programming**

เราจะใช้ Python (ภาษาที่ทุกคนเรียนมาแล้ว) จำลองพฤติกรรมของตัวแปรใน MCU
เพื่อเข้าใจ **เลขฐาน · overflow · two's complement · endianness** ก่อนไปเขียนภาษา C จริงในครั้งที่ 6

> วิธีใช้: กด `Shift + Enter` เพื่อรันทีละเซลล์ แล้วลองแก้ตัวเลขดูผลลัพธ์


## 1. เลขฐานใน Python
Python แปลงฐานให้ได้ในตัว ลองรันดู

In [ ]:
# prefix 0b = ฐานสอง, 0x = ฐานสิบหก
print(0b1101)        # ฐานสอง 1101 -> ?
print(0x2F)          # ฐานสิบหก 2F -> ?

# แปลงกลับ
print(bin(45))       # ฐานสิบ -> ฐานสอง
print(hex(254))      # ฐานสิบ -> ฐานสิบหก
print(int("11010110", 2))   # สตริงฐานสอง -> ฐานสิบ
print(int("C8", 16))        # สตริงฐานสิบหก -> ฐานสิบ

### ลองเอง 1
เติมโค้ดแปลง `10110101` (ฐานสอง) เป็นฐานสิบ และ `200` (ฐานสิบ) เป็นฐานสิบหก

In [ ]:
# เขียนคำตอบตรงนี้
print(int("10110101", 2))
print(hex(200))

## 2. Overflow — ทำไม MCU ถึง "วนกลับ"
ใน Python จำนวนเต็มโตได้ไม่จำกัด แต่ใน C ตัวแปรมีขนาดตายตัว เช่น `uint8_t` = 8 บิต (0..255)
เราจำลองด้วยการ mask ด้วย `& 0xFF` (เก็บแค่ 8 บิตล่าง)

In [ ]:
u = 250          # uint8_t
for _ in range(8):
    print(u)
    u = (u + 1) & 0xFF   # 255 + 1 -> วนกลับเป็น 0 !

**สังเกต:** พอเกิน 255 ค่าวนกลับเป็น 0 นี่คือ *overflow / wrap-around* — บั๊กยอดฮิตในงานฝังตัว
เช่น ตัวนับที่ลืมเผื่อขนาด จะรีเซ็ตเองโดยไม่ตั้งใจ

### ลองเอง 2
เปลี่ยน `0xFF` เป็น `0xFFFF` (16 บิต) แล้วดูว่า overflow ที่ค่าเท่าใด

In [ ]:
u = 65530
for _ in range(8):
    print(u)
    u = (u + 1) & 0xFFFF

## 3. Two's Complement — เก็บเลขลบยังไง
MCU เก็บเลขลบด้วยวิธี two's complement: `invert ทุกบิต แล้ว +1`
ตัวแปร `int8_t` เก็บได้ -128..127

In [ ]:
def to_int8(bits_value):
    """ตีความค่า 8 บิตแบบมีเครื่องหมาย (two's complement)"""
    v = bits_value & 0xFF
    return v - 256 if v > 127 else v

# หา two's complement ของ -40
neg = (-40) & 0xFF
print("bit pattern ของ -40 :", format(neg, '08b'))   # 11011000

# ตีความบิตกลับเป็นค่า
print("11111000 =", to_int8(0b11111000))             # -8
print("10000001 =", to_int8(0b10000001))             # -127

### int8_t ก็ overflow ได้ — แต่กระโดดเป็นค่าลบ

In [ ]:
s = 124
for _ in range(6):
    print(s)
    s = to_int8(s + 1)     # 127 + 1 -> กระโดดเป็น -128 !

### ลองเอง 3
ใช้ `to_int8` หาว่าบิต `11101100` (ค่าจากเซนเซอร์อุณหภูมิ) คือกี่องศา

In [ ]:
print(to_int8(0b11101100))   # ?

## 4. Endianness — ไบต์ไหนมาก่อน
เมื่อเก็บเลข 4 ไบต์ (เช่น `0x12345678`) ลงหน่วยความจำ ไบต์ไหนอยู่ address แรก?
- **little-endian**: ไบต์ต่ำสุดก่อน (ESP32, x86 เป็นแบบนี้)
- **big-endian**: ไบต์สูงสุดก่อน (network order)

In [ ]:
import struct
x = 0x12345678
print("little-endian:", struct.pack('<I', x).hex())   # 78563412  (แบบ ESP32)
print("big-endian   :", struct.pack('>I', x).hex())   # 12345678

# ดูทีละไบต์แบบ little-endian
for i, b in enumerate(struct.pack('<I', x)):
    print(f"address+{i} : 0x{b:02X}")

### ลองเอง 4
เปลี่ยนค่า `x` เป็น `0xAABBCCDD` แล้วดูลำดับไบต์แบบ little-endian

In [ ]:
x = 0xAABBCCDD
print(struct.pack('<I', x).hex())   # ?

## 5. เชื่อมไปภาษา C
โค้ด Python ที่เราเขียนวันนี้ ตรงกับ C แทบบรรทัดต่อบรรทัด — ต่างที่ C ต้องประกาศ *ชนิดและขนาด* ตัวแปรเอง

| Python (วันนี้) | C (ครั้งที่ 6) |
|---|---|
| `u = 250` | `uint8_t u = 250;` |
| `u = (u + 1) & 0xFF` | `u = u + 1;`  *(overflow อัตโนมัติ)* |
| `to_int8(v)` | `int8_t v;`  *(ตีความให้อัตโนมัติ)* |
| `x & 1` | `x & 1`  *(เหมือนกันเป๊ะ)* |

**ใจความ:** ใน C ขนาดตัวแปรคือสิ่งที่เรากำหนดเอง — overflow และ signed/unsigned จึงเป็นเรื่องที่ต้องระวังตลอดเวลาในงานฝังตัว


---
### แบบฝึกท้าย notebook (ทำในคาบ)
1. เขียนฟังก์ชัน `to_binary(n)` แปลงเลขฐานสิบเป็นสตริงฐานสอง โดย**ไม่ใช้** `bin()`
2. จำลอง `uint8_t` counter ที่นับ 0→255 แล้ววนกลับ 0 พิมพ์ครบ 260 ค่า

In [ ]:
# พื้นที่ทำแบบฝึก
def to_binary(n):
    if n == 0:
        return "0"
    result = ""
    while n > 0:
        result = str(n % 2) + result
        n //= 2
    return result

print(to_binary(200))   # 11001000